# ANALIZA DANYCH Z ZEGARKA APPLE WATCH


## Podstawowe informacje o pliku:

* Plik jest pobierany z aplikacji Zdrowie (Apple Health) na telefonie
* format: XML
* struktura: hierarchiczna, z wieloma różnymi typami rekordów (Record, Correlation, Workout, ActivitySummary, ClinicalRecord, Audiogram, VisionPrescription)
* zawiera dane dotyczące zdrowia, aktywności, snu, tętna, kroków, itp.
* jest dość duży ponad 700 mb





Me
*   HKCharacteristicTypeIdentifierDateOfBirth
*   HKCharacteristicTypeIdentifierBiologicalSex
*   HKCharacteristicTypeIdentifierBloodType
*   HKCharacteristicTypeIdentifierFitzpatrickSkinType
*   HKCharacteristicTypeIdentifierCardioFitnessMedicationsUse

Record
*   type
*   unit
*   value
*   sourceName
*   sourceVersion
*   device
*   creationDate
*   startDate
*   ~~endDate~~~~~~

Workout ((MetadataEntry|WorkoutEvent|WorkoutRoute|WorkoutStatistics)*)
Workout
*   workoutActivityType
*   duration
*   durationUnit
*   totalDistance
*   totalDistanceUnit
*   totalEnergyBurned
*   totalEnergyBurnedUnit
*   sourceName
*   sourceVersion
*   device
*   creationDate
*   startDate
*   endDate

WorkoutActivity
*   uuid
*   startDate
*   endDate
*   duration
*   durationUnit

WorkoutEvent
*   type
*   date
*   duration
*   durationUnit

WorkoutStatistics
*   type
*   startDate
*   endDate
*   average
*   minimum
*   maximum
*   sum
*   unit

WorkoutRoute
*   sourceName
*   sourceVersion
*   device
*   creationDate
*   startDate
*   endDate


ActivitySummary
*   dateComponents
*   activeEnergyBurned
*   activeEnergyBurnedGoal
*   activeEnergyBurnedUnit
*   appleMoveTime
*   appleMoveTimeGoal
*   appleExerciseTime
*   appleExerciseTimeGoal
*   appleStandHours
*   appleStandHoursGoal

MetadataEntry
  key   
  value 

<!-- Note: Heart Rate Variability records captured by Apple Watch may include an associated list of instantaneous beats-per-minute readings. -->
<!ELEMENT HeartRateVariabilityMetadataList (InstantaneousBeatsPerMinute*)>
<!ELEMENT InstantaneousBeatsPerMinute EMPTY>
<!ATTLIST InstantaneousBeatsPerMinute
  bpm  
  time 
>

## TO DO:
1. [x] rozgryźć strukturę pliku xml - pól w nim zawartych - co dokładnie w nim jest
2. [x] sprawdzić pomiar EKG i możliwość ściągnięcia danych
3. [x] napisać kod pythona do wczytania tych danych
4. [x] zastanowić się nad programem ćwiczeń dla osób o różnej kondycji, z pomiarem zegarkiem

In [39]:
import os
import streamlit as st
import time
from lxml import etree as ET
import pandas as pd


In [40]:
path = "eksport.xml"

In [41]:
def discover_xml_structure(file_path):
    # Słownik: Klucz = Nazwa Tagu, Wartość = Zbiór atrybutów 'type' lub 'workoutActivityType'
    found_structures = {}

    context = ET.iterparse(file_path, events=("end",))

    for event, elem in context:
        tag_name = elem.tag

        #  dodajemy do slownika jesli nie ma w nim jeszcze
        if tag_name not in found_structures:
            found_structures[tag_name] = set()

        # sprawdzamy podtypy głównych tagów
        sub_type = elem.get("type") or elem.get("workoutActivityType") or "Brak podtypu"

        found_structures[tag_name].add(sub_type)

        # Czyścimy RAM
        elem.clear()

    return found_structures

moje_dane = discover_xml_structure(path)

print("ODKRYTE TAGI + typy danych")
for tag, sub_types in moje_dane.items():
    print(f"\n TAG: <{tag}>")
    print(f"   Liczba różnych typów: {len(sub_types)}")
    for st in list(sub_types):
        print(f"     - {st}")

ODKRYTE TAGI + typy danych

 TAG: <ExportDate>
   Liczba różnych typów: 1
     - Brak podtypu

 TAG: <Me>
   Liczba różnych typów: 1
     - Brak podtypu

 TAG: <Record>
   Liczba różnych typów: 41
     - HKQuantityTypeIdentifierAppleSleepingWristTemperature
     - HKQuantityTypeIdentifierHeartRateRecoveryOneMinute
     - HKQuantityTypeIdentifierWalkingHeartRateAverage
     - HKQuantityTypeIdentifierAppleWalkingSteadiness
     - HKQuantityTypeIdentifierHeartRate
     - HKCategoryTypeIdentifierAppleStandHour
     - HKQuantityTypeIdentifierTimeInDaylight
     - HKQuantityTypeIdentifierActiveEnergyBurned
     - HKQuantityTypeIdentifierBodyFatPercentage
     - HKQuantityTypeIdentifierSixMinuteWalkTestDistance
     - HKQuantityTypeIdentifierWalkingSpeed
     - HKCategoryTypeIdentifierAudioExposureEvent
     - HKQuantityTypeIdentifierBasalEnergyBurned
     - HKQuantityTypeIdentifierWalkingAsymmetryPercentage
     - HKQuantityTypeIdentifierHeartRateVariabilitySDNN
     - HKQuantityTypeIdentifi

In [44]:
important_tags = moje_dane.keys()
print(important_tags)


dict_keys(['ExportDate', 'Me', 'Record', 'MetadataEntry', 'WorkoutEvent', 'WorkoutStatistics', 'Workout', 'FileReference', 'WorkoutRoute', 'ActivitySummary', 'InstantaneousBeatsPerMinute', 'HeartRateVariabilityMetadataList', 'HealthData'])


In [43]:
print(moje_dane)

{'ExportDate': {'Brak podtypu'}, 'Me': {'Brak podtypu'}, 'Record': {'HKQuantityTypeIdentifierAppleSleepingWristTemperature', 'HKQuantityTypeIdentifierHeartRateRecoveryOneMinute', 'HKQuantityTypeIdentifierWalkingHeartRateAverage', 'HKQuantityTypeIdentifierAppleWalkingSteadiness', 'HKQuantityTypeIdentifierHeartRate', 'HKCategoryTypeIdentifierAppleStandHour', 'HKQuantityTypeIdentifierTimeInDaylight', 'HKQuantityTypeIdentifierActiveEnergyBurned', 'HKQuantityTypeIdentifierBodyFatPercentage', 'HKQuantityTypeIdentifierSixMinuteWalkTestDistance', 'HKQuantityTypeIdentifierWalkingSpeed', 'HKCategoryTypeIdentifierAudioExposureEvent', 'HKQuantityTypeIdentifierBasalEnergyBurned', 'HKQuantityTypeIdentifierWalkingAsymmetryPercentage', 'HKQuantityTypeIdentifierHeartRateVariabilitySDNN', 'HKQuantityTypeIdentifierOxygenSaturation', 'HKQuantityTypeIdentifierRestingHeartRate', 'HKDataTypeSleepDurationGoal', 'HKQuantityTypeIdentifierWalkingDoubleSupportPercentage', 'HKQuantityTypeIdentifierRespiratoryRate'

In [1]:

diff_types = set()

diff_tags = set()
# Iterujemy po pliku
for event, elem in ET.iterparse(path, events=("end",)):
    if elem.tag in important_tags:
        typ = elem.get("type") or elem.tag or elem.get("workoutActivityType") or "Brak podtypu"

        if typ not in diff_types:
            print(f"\n TYP: {typ}")
            print(f"   Atrybuty: {elem.attrib}")
            diff_types.add(typ)


            children = list(elem)
            if children:
                for child in children:
                    print(f"   typ: <{child.tag}> atrybuty: {child.attrib}")




        elem.clear()


NameError: name 'ET' is not defined

Wszystkie typy danych w moim pliku:
{'HKQuantityTypeIdentifierAppleSleepingWristTemperature', 'HKQuantityTypeIdentifierHeartRateRecoveryOneMinute', 'Me', 'HKQuantityTypeIdentifierWalkingHeartRateAverage', 'HKWorkoutEventTypeResume', 'HKQuantityTypeIdentifierAppleWalkingSteadiness', 'HKQuantityTypeIdentifierHeartRate', 'HKCategoryTypeIdentifierAppleStandHour', 'HKQuantityTypeIdentifierTimeInDaylight', 'HKQuantityTypeIdentifierActiveEnergyBurned', 'HKQuantityTypeIdentifierBodyFatPercentage', 'HKQuantityTypeIdentifierSixMinuteWalkTestDistance', 'ActivitySummary', 'HKQuantityTypeIdentifierWalkingSpeed', 'HKCategoryTypeIdentifierAudioExposureEvent', 'Workout', 'HKWorkoutEventTypePause', 'HKQuantityTypeIdentifierBasalEnergyBurned', 'HKQuantityTypeIdentifierWalkingAsymmetryPercentage', 'HKWorkoutEventTypeSegment', 'HKQuantityTypeIdentifierHeartRateVariabilitySDNN', 'HKQuantityTypeIdentifierOxygenSaturation', 'HKQuantityTypeIdentifierRestingHeartRate', 'HKDataTypeSleepDurationGoal', 'InstantaneousBeatsPerMinute', 'HKQuantityTypeIdentifierWalkingDoubleSupportPercentage', 'HKQuantityTypeIdentifierRespiratoryRate', 'HKQuantityTypeIdentifierAppleStandTime', 'HKQuantityTypeIdentifierPhysicalEffort', 'HKQuantityTypeIdentifierAppleExerciseTime', 'HKQuantityTypeIdentifierFlightsClimbed', 'HKQuantityTypeIdentifierStairDescentSpeed', 'HKQuantityTypeIdentifierHeadphoneAudioExposure', 'HKCategoryTypeIdentifierHighHeartRateEvent', 'HKQuantityTypeIdentifierVO2Max', 'HKCategoryTypeIdentifierHeadphoneAudioExposureEvent', 'HKCategoryTypeIdentifierMenstrualFlow', 'HKQuantityTypeIdentifierStepCount', 'HKQuantityTypeIdentifierEnvironmentalSoundReduction', 'HKQuantityTypeIdentifierEnvironmentalAudioExposure', 'HKQuantityTypeIdentifierDistanceWalkingRunning', 'HKQuantityTypeIdentifierWaistCircumference', 'HKQuantityTypeIdentifierStairAscentSpeed', 'HKQuantityTypeIdentifierBodyMass', 'HKQuantityTypeIdentifierHeight', 'HKQuantityTypeIdentifierDistanceCycling', 'HKCategoryTypeIdentifierSleepAnalysis', 'HKQuantityTypeIdentifierWalkingStepLength', 'HKWorkoutEventTypeMarker'}

In [37]:
#Stworzenie dataframe z wybranym typem danych

short_names = {
    # --- CIAŁO / PODSTAWOWE ---
    'HKQuantityTypeIdentifierHeight': 'height',
    'HKQuantityTypeIdentifierBodyMass': 'weight',
    'HKQuantityTypeIdentifierWaistCircumference': 'waist',
    'HKQuantityTypeIdentifierBodyFatPercentage': 'fat_pct',

    # --- SERCE (HEART) ---
    'HKQuantityTypeIdentifierHeartRate': 'hr',
    'HKQuantityTypeIdentifierRestingHeartRate': 'rhr',
    'HKQuantityTypeIdentifierWalkingHeartRateAverage': 'hr_walk_avg',
    'HKQuantityTypeIdentifierHeartRateVariabilitySDNN': 'hrv',
    'HKQuantityTypeIdentifierHeartRateRecoveryOneMinute': 'hrr_1min',
    'HKCategoryTypeIdentifierHighHeartRateEvent': 'hr_high_event',

    # --- AKTYWNOŚĆ / ENERGIA ---
    'HKQuantityTypeIdentifierStepCount': 'steps',
    'HKQuantityTypeIdentifierDistanceWalkingRunning': 'dist_walk',
    'HKQuantityTypeIdentifierDistanceCycling': 'dist_cycle',
    'HKQuantityTypeIdentifierBasalEnergyBurned': 'kcal_basal',
    'HKQuantityTypeIdentifierActiveEnergyBurned': 'kcal_active',
    'HKQuantityTypeIdentifierFlightsClimbed': 'flights',
    'HKQuantityTypeIdentifierAppleExerciseTime': 'exercise_min',
    'HKQuantityTypeIdentifierAppleStandTime': 'stand_min',
    'HKCategoryTypeIdentifierAppleStandHour': 'stand_hr',
    'HKQuantityTypeIdentifierPhysicalEffort': 'effort',
    'HKQuantityTypeIdentifierVO2Max': 'vo2max',

    # --- MOBILNOŚĆ / CHÓD ---
    'HKQuantityTypeIdentifierWalkingSpeed': 'walk_speed',
    'HKQuantityTypeIdentifierWalkingStepLength': 'walk_step_len',
    'HKQuantityTypeIdentifierWalkingAsymmetryPercentage': 'walk_asym',
    'HKQuantityTypeIdentifierWalkingDoubleSupportPercentage': 'walk_dbl_supp',
    'HKQuantityTypeIdentifierStairAscentSpeed': 'stair_up_speed',
    'HKQuantityTypeIdentifierStairDescentSpeed': 'stair_down_speed',
    'HKQuantityTypeIdentifierAppleWalkingSteadiness': 'walk_steadiness',
    'HKQuantityTypeIdentifierSixMinuteWalkTestDistance': 'walk_6min',

    # --- ODDECH / POZIOMY ---
    'HKQuantityTypeIdentifierOxygenSaturation': 'spo2',
    'HKQuantityTypeIdentifierRespiratoryRate': 'resp_rate',

    # --- SEN ---
    'HKCategoryTypeIdentifierSleepAnalysis': 'sleep',
    'HKDataTypeSleepDurationGoal': 'sleep_goal',
    'HKQuantityTypeIdentifierAppleSleepingWristTemperature': 'wrist_temp',

    # --- ŚRODOWISKO / DŹWIĘK ---
    'HKQuantityTypeIdentifierEnvironmentalAudioExposure': 'audio_env',
    'HKQuantityTypeIdentifierHeadphoneAudioExposure': 'audio_hp',
    'HKQuantityTypeIdentifierEnvironmentalSoundReduction': 'sound_red',
    'HKCategoryTypeIdentifierAudioExposureEvent': 'audio_event',
    'HKCategoryTypeIdentifierHeadphoneAudioExposureEvent': 'audio_hp_event',
    'HKQuantityTypeIdentifierTimeInDaylight': 'daylight',

    # --- INNE ---
    'HKCategoryTypeIdentifierMenstrualFlow': 'menstr'
}


In [38]:


def load_all_health_data(file_path, mapping):
    """
    Funkcja zwraca słownik DataFrameów z krótkimi nazwami, żeby szybciej wpisywać.
    """
    if file_path is None:
        return "Zła ścieżka pliku, spróbuj ponownie"
    if mapping is None:
        mapping = types_list # jeśli nie mamy krótkich nazw używamy pobranych długich

    # 1. Tworzymy puste listy dla każdego krótkiego skrótu ze słownika
    data_buckets = {short_name: [] for short_name in mapping.values()}

    # 2. Startujemy iterparse
    # event="end" oznacza, że reagujemy, gdy zamknie się tag </Record>
    context = ET.iterparse(file_path, events=("end",))

    for event, elem in context:
        if elem.tag == "Record" or elem.tag == "Workout" or :
            long_type = elem.get("type")

            # 3. Jeśli ten typ jest w naszym słowniku krótkich nazw
            if long_type in mapping:
                short_name = mapping[long_type]

                # Wyciągamy dane do słownika (lekki format)
                record = {
                    'value': elem.get('value'),
                    'unit': elem.get('unit'),
                    'sourceName': elem.get('sourceName'),
                    'startDate': elem.get('startDate'),
                    'endDate': elem.get('endDate'),

                }

                for child in elem:
                    if child.tag == "MetadataEntry":
                        record[child.get('key')] = child.get('value')

                data_buckets[short_name].append(record)

            elem.clear()



        root = context.root

    # 4. Zamieniamy listy na DataFrames i naprawiamy typy
    final_dfs = {}
    print("📊 Konwertuję dane na tabele...")

    for short_name, records in data_buckets.items():
        if records: # Tworzymy DF tylko jeśli są dane
            df = pd.DataFrame(records)

            # Automatyczna konwersja na liczby i daty
            if 'value' in df.columns:
                df['value'] = pd.to_numeric(df['value'], errors='coerce')
            if 'startDate' in df.columns:
                df['startDate'] = pd.to_datetime(df['startDate'])
            if 'endDate' in df.columns:
                df['endDate'] = pd.to_datetime(df['endDate'])

            final_dfs[short_name] = df

    print("✅ Wszystkie dane wczytane!")
    return final_dfs



hd = load_all_health_data("eksport.xml", short_names)


df = hd.get('hr')
df.head()
df.describe()
df_steps = hd.get('steps')

SyntaxError: invalid syntax (1622740882.py, line 18)

In [ ]:
print(df.head())
print(df.describe())
print(df_steps.head())

Czyścimy  dane - usuwamy kolumny w których wszystkie wartości sa nan

In [ ]:
for name, df in hd.items():
    # axis=1 oznacza kolumny, how='all' tylko te całkiem puste
    hd[name] = df.dropna(axis=1, how='all')

In [ ]:
for name in hd:
    hd[name].info()


In [ ]:
hd['hr'].info()
print(hd['hr'].describe())
print(hd['hr'].tail(20))

In [ ]:
df = hd['hr']


In [ ]:
# Usuwamy konkretne kolumny techniczne, których nie potrzebujemy do analizy tętna
df = hd['hr'].drop(columns=['HKMetadataKeySyncVersion', 'HKMetadataKeySyncIdentifier'], errors='ignore')

In [ ]:
print(df.columns)

In [ ]:


df['is_point'] = df['startDate'] == df['endDate']

# Szybki podgląd: ile masz przedziałów, a ile punktów?
print(df['is_point'].value_counts())

In [ ]:
# Usuwamy rekordy, które mają IDENTYCZNY start, koniec I wartość
df_clean = df.drop_duplicates(subset=['startDate', 'endDate', 'value'])

print(f"Usunięto {len(df) - len(df_clean)} idealnych duplikatów.")

In [ ]:
print(df.columns)

In [ ]:
#usuwamy duplikaty
df = df.drop_duplicates(subset=['startDate', 'endDate', 'value'])

# bierzemy dane pobierane tylko z zegarka
df = df[df['sourceName'].str.contains('Apple', na=False)]


# Wywalamy rekordy, gdzie koniec jest przed początkiem i jakieś błędne
df = df[df['endDate'] >= df['startDate']]

# indeksujemy po dacie poczatkowej
df = df.set_index('startDate')

df_hr_1min = df['value'].resample('1min').mean()

df_hr_1min = df_hr_1min.dropna()

print(f"Sukces! Twoje dane tętna są teraz jednolite. Zostało {len(df_hr_1min)} czystych minut.")

In [ ]:
display(df.tail(100))